# 13장 AI 에이전트

이 장의 예제와 실습을 실행하는 노트북입니다. 개념 설명은 HTML 교재 `13_AI_에이전트.html`에 있습니다. 위에서부터 차례로 실행합니다. 실습의 풀이는 교재의 실습 상자에 접혀 있습니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

장마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 터미널에서 실행하고 커널을 다시 시작합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(6장 "구조화 출력") |

**PC에서 모델을 내려받을 때(공개 LLM, 임베딩 모델, 리랭커)**

| 환경 변수 | 값 |
| --- | --- |
| `HF_ENDPOINT` | 사내 Hugging Face 프록시 주소. 비워 두면 huggingface.co에서 받습니다. 이 값도 Jupyter를 시작하기 전에 설정합니다 |
| `EMBED_MODEL` | 설정하지 않아도 됩니다. 다른 임베딩 모델을 쓰거나 모델 파일이 든 폴더를 따로 받았을 때, 모델 이름이나 폴더 경로를 넣습니다 |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정한 값은 모든 장의 노트북에 그대로 적용됩니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [ ]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

In [ ]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data'   # 재료 폴더. 장마다 쓰는 재료는 이 아래에 있습니다

# 설치: python -m pip install pypdf beautifulsoup4 transformers
# 13장과 관통 프로젝트 5단계에서 씁니다. 예제 13-1은 1장의 토크나이저만 씁니다(모델은 내려받지 않습니다)
import embed
from doc_tools import CATALOG, TOOLS, read_doc, search_docs   # 사내 문서 120건의 검색과 읽기, 도구 정의(노트북/doc_tools.py)

LOCAL_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
embed.ask_endpoint(LOCAL_MODEL)          # 토크나이저가 PC에 없고 HF_ENDPOINT가 비어 있으면 주소를 묻습니다
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)


# 관통 프로젝트 1, 2단계의 풀이가 쓰는 것입니다(5장, 6장의 준비 코드와 같습니다)
import csv
import random

with open(f'{DATA}/문의.csv', encoding='utf-8') as file:
    inquiries = list(csv.DictReader(file))        # 문의 60건. 열: 번호, 문의, 라벨, 긴급, 비고
LABELS = ['설비 이상', '품질', '안전', '문서와 데이터 요청']
PRICES = {'소형': {'input': 0.10, 'output': 0.50}, '기본': {'input': 2.00, 'output': 10.00},
          '대형': {'input': 4.00, 'output': 20.00}}   # 100만 토큰당 단가(달러). 5장과 같습니다(2026-11 확인)


def cost(reply, tier):
    """응답 하나의 요금(달러). 입력 토큰과 출력 토큰(thinking 포함)에 각각의 단가를 곱해 더합니다"""
    price = PRICES[tier]
    return (reply.input_tokens * price['input'] + reply.output_tokens * price['output']) / 1_000_000


print(f'문서 {len(CATALOG)}건, 도구 {[tool["name"] for tool in TOOLS]}, 토크나이저 {LOCAL_MODEL}')


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. "준비 (1)" 셀을 다시 실행해 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `ModuleNotFoundError: No module named '...'` | 패키지를 설치하지 않았습니다. "준비 (2)" 셀의 `# 설치:` 주석에 있는 명령을 터미널에서 실행하고 커널을 다시 시작합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 예제 13-1. 도구 정의는 입력에 들어가는 글이다

교재의 절: "도구 호출"

1장의 공개 모델 Qwen2.5의 채팅 템플릿에 같은 질문을 도구 없이, 도구 두 개와 함께 넣어 입력 토큰 수를 비교하고, 도구 정의가 들어간 입력 전체를 출력합니다. 이어서 모델이 도구를 요청하고 코드가 결과를 돌려준 뒤의 입력에서 새로 붙은 부분만 출력합니다. 도구 정의, 도구 호출, 도구 결과가 각각 어떤 글로 들어가는지 봅니다. API를 호출하지 않습니다.

In [ ]:
# Qwen의 템플릿은 OpenAI 형식의 도구 정의를 받으므로 TOOLS(Anthropic 형식)를 바꿔 넣습니다
qwen_tools = [{'type': 'function', 'function': {'name': tool['name'], 'description': tool['description'],
                                                'parameters': tool['input_schema']}} for tool in TOOLS]
question = [{'role': 'user', 'content': 'SSC가 뭔가요?'}]
plain = tokenizer.apply_chat_template(question, tokenize=False, add_generation_prompt=True)
with_tools = tokenizer.apply_chat_template(question, tools=qwen_tools, tokenize=False, add_generation_prompt=True)
print(f'입력 토큰: 도구 없이 {len(tokenizer(plain).input_ids)}개, 도구 두 개와 함께 {len(tokenizer(with_tools).input_ids)}개\n')
print(with_tools)

# 모델이 search_docs를 요청했고, 코드가 실행한 결과(앞 200자)를 돌려준 뒤의 입력입니다
turns = question + [{'role': 'assistant', 'content': '', 'tool_calls': [
                        {'type': 'function', 'function': {'name': 'search_docs', 'arguments': {'query': 'SSC'}}}]},
                    {'role': 'tool', 'content': search_docs('SSC')[:200]}]
after = tokenizer.apply_chat_template(turns, tools=qwen_tools, tokenize=False, add_generation_prompt=True)
print('---- 도구 호출과 결과가 붙은 뒤 새로 생긴 부분 ----')
print(after[len(with_tools) - len('<|im_start|>assistant\n'):])

입력 토큰: 도구 없이 38개, 도구 두 개와 함께 450개

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "search_docs", "description": "후공정 회사의 사내 문서 120건(작업 표준, 설비 매뉴얼, 품질 이슈 보고서, 회의록, 공지, 교육 자료, 안전 규정)을 키워드로 검색한다. 관련이 높은 순서로 5건의 문서 번호, 제목, 종류, 부서, 개정, 날짜, 상태(유효 또는 폐기)와 본문 한 줄을 돌려준다. 본문 전체는 돌려주지 않으므로 read_doc으로 읽는다. 검색어는 문서에 나올 법한 낱말로 짧게 쓴다.", "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "검색어. 예: 와이어 풀 강도 하한"}}, "required": ["query"]}}}
{"type": "function", "function": {"name": "read_doc", "description": "문서 번호로 사내 문서 하나의 본문을 읽는다. search_docs 결과에 나온 문서 번호만 쓴다.", "parameters": {"type": "object", "properties": {"doc_id": {"type": "string", "description": "문서 번호. 예: SOP-WB-001"}}, "required": ["doc_id"]}}}
</tools>

For each f

## 예제 13-2. 모델은 실행을 요청할 뿐이다

교재의 절: "도구 호출"

도구 정의 두 개와 함께 질문 세 개를 한 번씩 보내고, 도구를 실행하지 않은 채 응답만 출력합니다. 사내 약어를 묻는 질문, 널리 알려진 공정을 묻는 질문, 문서 번호를 지목한 질문에 종료 이유와 도구 호출이 어떻게 오는지 봅니다. 이 셀은 API를 3번 호출합니다.

In [ ]:
QUESTIONS = ['SSC가 뭔가요?',                                    # 사내 약어: 문서를 찾아야 답할 수 있음
             '와이어 본딩이 어떤 공정인지 한 문장으로 설명해 주세요.',   # 널리 알려진 일반 지식
             'QIR-2026-014 문서를 열어 원인만 알려 주세요.']             # 문서 번호를 직접 지목
for question in QUESTIONS:
    reply = llm.ask_messages([{'role': 'user', 'content': question}], tools=TOOLS, model='소형', max_tokens=1000)
    print(f'질문: {question}\n  종료 이유: {reply.stop_reason}, 입력 {reply.input_tokens:,}토큰')
    for call in reply.tool_calls:                  # 모델이 요청한 도구 호출. 코드는 아직 아무것도 실행하지 않았습니다
        print(f'  도구 호출: {call.name}({call.arguments}), id {call.id}')
    if reply.text:
        print('  응답 텍스트:', reply.text.strip().replace('\n', ' ')[:150])

질문: SSC가 뭔가요?
  종료 이유: tool_use, 입력 748토큰
  도구 호출: search_docs({'query': 'SSC'}), id toolu_01L349KSgTssyA35YzUjCAj7
질문: 와이어 본딩이 어떤 공정인지 한 문장으로 설명해 주세요.
  종료 이유: end, 입력 770토큰
  응답 텍스트: 와이어 본딩은 반도체 칩의 본드 패드와 리드프레임 또는 기판을 금·구리·알루미늄 등의 미세한 금속 와이어로 열, 압력, 초음파를 이용해 전기적으로 연결하는 후공정입니다.
질문: QIR-2026-014 문서를 열어 원인만 알려 주세요.
  종료 이유: tool_use, 입력 764토큰
  도구 호출: read_doc({'doc_id': 'QIR-2026-014'}), id toolu_015g8Wjnq9NjDYozwXiwTmGE


## 예제 13-3. 에이전트 루프

교재의 절: "에이전트 루프"

`run_agent()`는 종료 이유가 `tool_use`인 동안 도구를 실행해 결과를 붙이고 다시 요청하는 반복문입니다. 두 문서를 이어야 답할 수 있는 평가셋 질문 E40을 묻고, 단계마다 입력 토큰과 모델이 요청한 도구를 출력합니다. 모델이 정한 검색어, 한 응답에 든 도구 호출의 수, 단계마다 늘어나는 입력 토큰을 봅니다. 이 셀은 API를 3~5번 호출합니다.

In [ ]:
TOOL_FUNCTIONS = {'search_docs': search_docs, 'read_doc': read_doc}     # 도구 이름 → 실제로 실행할 함수
SYSTEM = '너는 후공정 회사의 사내 문서 도우미다. 도구로 사내 문서를 찾아 근거가 있는 것만 답하고, 답 끝에 근거 문서 번호를 적는다.'


def run_agent(question, max_steps=8):
    messages = [{'role': 'user', 'content': question}]
    for step in range(1, max_steps + 1):
        reply = llm.ask_messages(messages, system=SYSTEM, tools=TOOLS, model='소형', max_tokens=2000)
        print(f'{step}단계 입력 {reply.input_tokens:,}토큰:', [f'{call.name}({call.arguments})' for call in reply.tool_calls] or '최종 답')
        if reply.stop_reason != 'tool_use':                # 도구를 더 요청하지 않으면 응답 텍스트가 답입니다
            return reply.text
        if step == max_steps:                              # 결과를 돌려줄 다음 요청이 없으므로 도구도 실행하지 않습니다
            break
        messages.append(llm.assistant_turn(reply))
        for call in reply.tool_calls:                      # 요청받은 도구를 모두 실행하고, 결과에 같은 id를 붙입니다
            messages.append(llm.tool_result(call, TOOL_FUNCTIONS[call.name](**call.arguments)))
    return '(단계 상한에 닿아 멈췄습니다)'


print(run_agent('고객사 B의 요구로 강화된 기준 두 가지는 무엇이고 각각 얼마인가요?'))

1단계 입력 851토큰: ["search_docs({'query': '고객사 B 강화 기준'})", "search_docs({'query': '고객사 B 요구 기준'})"]
2단계 입력 2,193토큰: ["read_doc({'doc_id': 'SOP-WB-001'})", "read_doc({'doc_id': 'SOP-VI-001'})"]
3단계 입력 4,175토큰: 최종 답
고객사 B의 요구로 강화된 기준은 두 가지입니다.

1. **와이어 풀 강도 하한(금선 25μm): 5.0gf → 6.0gf**
   - 2026-03-16 개정 3에서 상향되었습니다(구리선은 6.0gf → 7.0gf).
   - 풀 테스트는 로트마다 5와이어를 측정하며, 한 와이어라도 하한에 못 미치면 로트를 홀드합니다.

2. **리드 코플래너리티: 0.10mm → 0.08mm 초과 시 불량**
   - 2026-10-05 개정 4에서 강화되었습니다.
   - 적용 제품은 QX48, SP8입니다.

근거 문서: SOP-WB-001(와이어 본딩 작업 표준, 개정 3), SOP-VI-001(외관 검사 기준, 개정 4)


## 실습 13-1. 도구 정의가 도구 선택을 바꾼다

교재의 절: "개념 확인 실습"

문서 내용을 찾는 `search_docs`와, 문서 목록을 종류, 부서, 상태 조건으로 필터링해 세는 `list_docs` 두 도구를 줍니다. 이름, 설명, 인자가 모두 모호한 정의(`VAGUE`)와 구체적인 정의(`CLEAR`)로 질문 6개를 보내고, 모델이 처음 요청한 도구가 기대한 도구와 같은 질문 수를 셉니다. 도구는 실행하지 않습니다. `list_docs`의 구체적인 설명과 `first_tool()`을 작성하고, 두 정의의 맞은 수를 먼저 `my_guess`에 적습니다. 결과를 보고, 두 정의에서 모델이 무엇을 단서로 도구를 골랐는지 한 문장으로 설명하면 끝납니다. 이 셀은 질문 6개를 두 정의로 한 번씩 보내므로 API를 12번 호출합니다.

In [ ]:
LIST_SCHEMA = {'type': 'object', 'properties': {
    'doc_type': {'type': 'string', 'description': '문서 종류. 예: 설비 매뉴얼'},
    'dept': {'type': 'string', 'description': '관리 부서. 예: 조립기술팀'},
    'status': {'type': 'string', 'enum': ['유효', '폐기']}}}
GENERIC = {'type': 'object', 'properties': {'text': {'type': 'string'}}, 'required': ['text']}
VAGUE = [{'name': 'doc_tool_a', 'description': '문서 도구', 'input_schema': GENERIC},      # 이름 말고는 두 정의가 같습니다
         {'name': 'doc_tool_b', 'description': '문서 도구', 'input_schema': GENERIC}]
CLEAR = [TOOLS[0], {'name': 'list_docs', 'description': '', 'input_schema': LIST_SCHEMA}]  # TODO: list_docs의 설명
CASES = [('다이 어태치 페이스트는 해동한 뒤 몇 시간 안에 다 써야 하나요?', 0),     # (질문, 기대한 도구의 번호)
         ('금선 와이어 풀 강도 하한은 얼마인가요?', 0),
         ('11월 테스트 핸들러 정기 PM은 언제 하나요?', 0),
         ('조립기술팀이 관리하는 유효한 설비 매뉴얼은 몇 건인가요?', 1),
         ('폐기된 구판 문서는 모두 몇 건인가요?', 1),
         ('환경안전팀이 관리하는 안전 규정 목록을 보여 주세요.', 1)]


def first_tool(question, tools):
    """질문을 tools와 함께 한 번 보내고, 모델이 처음 요청한 도구의 이름을 돌려줍니다. 도구를 요청하지 않았으면 '(바로 답함)'"""
    return ''  # TODO


my_guess = {'VAGUE': None, 'CLEAR': None}  # TODO: 6개 가운데 맞을 개수를 먼저 적습니다
for label, tools in [('VAGUE', VAGUE), ('CLEAR', CLEAR)]:
    chosen = [first_tool(question, tools) for question, _ in CASES]
    right = sum(name == tools[index]['name'] for name, (_, index) in zip(chosen, CASES))
    print(f'{label}: {right}/6 (예측 {my_guess[label]})', chosen)

## 실습 13-2. 단계 상한과 사람 승인

교재의 절: "개념 확인 실습"

예제 13-3의 `run_agent()`에 가드레일 셋을 더한 `run_guarded()`를 작성합니다. 설비 작업 요청을 등록하는 쓰기 도구 `request_work`를 더하되, 이 도구는 실행 전에 `approve()`를 거쳐 승인된 경우에만 실행합니다. 도구에서 예외가 발생하면 오류 내용을 `is_error=True`인 도구 결과로 돌려주고, `max_steps`에 닿으면 멈춥니다. 질문 두 개로 확인합니다. 하나는 문서를 찾고 작업 요청까지 시키는 질문이고, 하나는 문서에 답이 없는 질문(평가셋 E57)을 단계 상한 2로 묻는 것입니다. 모델이 `request_work`를 요청할지, 거절당한 뒤 무엇이라고 답할지, E57이 상한 안에 끝날지를 먼저 `my_guess`에 적습니다. 실제로 등록된 요청(`registered`)이 비어 있으면 끝납니다. 이 셀은 API를 최대 8번 호출합니다. 단계마다 한 번씩 부르므로 첫 질문은 단계 상한 6까지, E57은 2까지 부릅니다. 교재의 실행에서는 6번(첫 질문 4번, E57 2번)이었습니다.

In [ ]:
WORK_TOOL = {'name': 'request_work',
             'description': '설비보전팀에 설비 점검이나 수리 작업 요청을 등록한다. 등록하면 작업자가 배정되고 설비 일정이 바뀐다.',
             'input_schema': {'type': 'object', 'required': ['equipment', 'content'], 'properties': {
                 'equipment': {'type': 'string', 'description': '설비 번호. 예: W-07'},
                 'content': {'type': 'string', 'description': '요청 내용'}}}}
registered = []                                   # 실제로 등록된 작업 요청


def request_work(equipment, content):
    registered.append({'설비': equipment, '내용': content})
    return f'{equipment} 작업 요청을 등록했습니다.'


def approve(call):
    """사람에게 승인을 묻습니다. 실제로는 input()이나 화면의 버튼으로 묻습니다. 이 실습에서는 모두 거절합니다"""
    print(f'  [승인 요청] {call.name}({call.arguments}) → 거절')
    return False


GUARDED_FUNCTIONS = {**TOOL_FUNCTIONS, 'request_work': request_work}
WRITE_TOOLS = {'request_work'}                    # 실행 전에 사람의 승인을 거칠 도구


def run_guarded(question, max_steps=6):
    """예제 13-3의 run_agent에 사람 승인, 오류 결과, 단계 상한을 더합니다. (답, 사용한 단계 수)를 돌려줍니다"""
    return '', 0  # TODO


my_guess = {'request_work를 요청': None, '거절 뒤 답': None, 'E57이 2단계 안에 끝남': None}  # TODO
answer, steps = run_guarded('W-07에서 NSOP 알람이 반복됩니다. 관련 품질 이슈 보고서를 찾아 원인을 알려 주고, '
                            '설비보전팀에 W-07 점검 작업 요청을 등록해 주세요.')
print(f'답({steps}단계):', answer, '\n등록된 요청:', registered)
answer, steps = run_guarded('레이저 마킹기의 레이저 파장은 몇 nm인가요?', max_steps=2)
print(f'\n답({steps}단계):', answer)